# Species Occurrence Records, Distribution Models, and Conservation Metrics for Bats in Sub-Saharan Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset for bat species occurrence records in sub-Saharan Africa with the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.8499-ept6/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.8499-ept6/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Published date:", getattr(metadata, 'datePublished', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list the record sets and their associated field `@id` values for exploration.

In [ ]:
# List all record sets with their @id and name
for recordset in dataset.record_sets():
    print(f"RecordSet @id: {recordset.id}")
    print(f"  Name: {getattr(recordset, 'name', '<no name>')}")
    print(f"  Description: {getattr(recordset, 'description', '<no description>')}")
    print(f"  Fields:")
    for field in recordset.fields:
        print(f"    - Field @id: {field.id} | Name: {getattr(field, 'name', '<no name>')} | DataType: {getattr(field, 'dataType', '<unknown>')}")
    print("")

## 3. Data Extraction
Load data from the primary record set into a DataFrame for analysis. All entities (record set, fields, columns) are referenced by their `@id`.

We select the main occurrence records record set for further exploration.

In [ ]:
# Find the record set related to bat occurrence records
occurrence_recordset_id = None
for recordset in dataset.record_sets():
    if "Occurrence" in getattr(recordset, 'name', '') or "occurrence" in getattr(recordset, 'name', '').lower():
        occurrence_recordset_id = recordset.id
        break
# If not found, fallback to first record set
if occurrence_recordset_id is None:
    occurrence_recordset_id = dataset.record_sets()[0].id

record_sets = [occurrence_recordset_id]
dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

print("Columns in the occurrence DataFrame:", dataframes[occurrence_recordset_id].columns.tolist())
dataframes[occurrence_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalizing, and grouping. Reference all field names by their `@id`.

We select a numeric field (e.g., Area of Occupancy, Extent of Occurrence, or Elevation) and a grouping field (e.g., species `@id` or country).

In [ ]:
# Identify candidate numeric and group fields by their @id

# Retrieve field details from record set
selected_record_set = None
for recordset in dataset.record_sets():
    if recordset.id == occurrence_recordset_id:
        selected_record_set = recordset
        break

# Look for numeric fields (Float/Integer), and a grouping field (species or country)
numeric_field_id = None
group_field_id = None
for field in selected_record_set.fields:
    dtype = getattr(field, 'dataType', '').lower()
    if dtype in ['float', 'integer'] and not numeric_field_id:
        numeric_field_id = field.id
    if ("species" in getattr(field, 'name', '').lower() or "country" in getattr(field, 'name', '').lower()) and not group_field_id:
        group_field_id = field.id

# Fallback if not found
if numeric_field_id is None:
    # Try taking the first numeric column in DataFrame
    for col in dataframes[occurrence_recordset_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[occurrence_recordset_id][col]):
            numeric_field_id = col
            break

if group_field_id is None:
    # Try taking the first non-numeric column
    for col in dataframes[occurrence_recordset_id].columns:
        if not pd.api.types.is_numeric_dtype(dataframes[occurrence_recordset_id][col]):
            group_field_id = col
            break

# EDA - filter, normalize, group
df = dataframes[occurrence_recordset_id]
if numeric_field_id in df.columns:
    # Threshold for numeric field
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

    # Grouping
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we plot the filtered numeric field and grouped means by chosen group field using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Visualize group means
    if group_field_id in df.columns:
        grouped_df = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df.head(20))
        plt.xticks(rotation=60)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (top 20)")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()


## 6. Conclusion
This notebook demonstrated exploration of the FAIR^2 bat dataset using the Croissant schema and `mlcroissant`. We loaded metadata, examined available record sets and fields via their `@id`, extracted data into DataFrames, applied common EDA patterns, visualized distributions and group statistics, and provided a reproducible workflow for researchers.

For more advanced analyses, refer to the record set and field `@id` references from earlier code to target specific entities, and extend this workflow as needed for your research.